<div style="background:#0A005A;border-radius:18px;padding:28px 32px;color:#fff;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">

<div style="margin-top:22px"><span style="background:#613EFF;color:#fff;padding:5px 12px;border-radius:999px;font-size:.8em;font-weight:700;letter-spacing:.06em">TUTORIAL · NOTEBOOK</span>
<span style="color:#C9C2FF;font-size:.85em;margin-left:8px">Engenharia de Dados · Nível 4 · Ingestão, orquestração e transformação</span></div>
<h1 style="color:#fff;margin:14px 0 8px;font-size:2.1em;line-height:1.15">Prática: gravar a coleta em bancos de dados</h1>
<p style="color:#E4E0FF;margin:0 0 12px;font-size:1.05em">Grave os 61 registros do PNCP de quatro jeitos e veja por que o repositório tem um cliente próprio para gravar.</p>
<span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">⏱ cerca de 30 minutos</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">🐍 Python 3.10+</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">🐘 PostgreSQL da aplicação no ar</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">🗂️ arquivo do notebook anterior</span>
</div>

## O que você vai construir

| Parte | Ferramenta | Destino | O que observar |
|---|---|---|---|
| A | `psycopg2`, SQL escrito por você | PostgreSQL | Controle total e upsert pela chave |
| B | `pandas.to_sql` | PostgreSQL | Rápido, mas duplica e não grava listas |
| C | `ClientPostgresDB.insert_data`, do repositório | PostgreSQL | O jeito que as DAGs do Gov Hub gravam |
| D | `sqlite3` | Arquivo SQLite | O mesmo padrão em outro banco |

<table style="border-collapse:collapse;width:100%;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><thead><tr style="background:#F2F1F6"><th style="width:56px"></th><th style="text-align:left;padding:8px 10px;width:36px">#</th><th style="text-align:left;padding:8px 10px">Passo</th><th style="text-align:left;padding:8px 10px">O que você vai ver</th></tr></thead><tbody><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">1</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">Conectar ao PostgreSQL</td><td style="padding:8px 10px;color:#444;text-align:left">endereço, usuário e senha sem expor nada</td></tr><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">2</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">psycopg2 com upsert</td><td style="padding:8px 10px;color:#444;text-align:left">a chave que impede duplicatas</td></tr><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">3</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">pandas.to_sql</td><td style="padding:8px 10px;color:#444;text-align:left">rápido, mas duplica e não grava listas</td></tr><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">4</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">O cliente do repositório</td><td style="padding:8px 10px;color:#444;text-align:left">o insert_data que as DAGs usam</td></tr><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">5</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">Outro banco: SQLite</td><td style="padding:8px 10px;color:#444;text-align:left">o mesmo padrão em um arquivo</td></tr><tr><td style="padding:8px 10px;text-align:center"></td><td style="padding:8px 10px;color:#BE006E;font-weight:700;text-align:left">6</td><td style="padding:8px 10px;color:#0A005A;font-weight:700;text-align:left">Conferir tudo</td><td style="padding:8px 10px;color:#444;text-align:left">as quatro formas lado a lado</td></tr></tbody></table>

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">⚠️ Antes de começar</b><br>Suba o banco da aplicação: na raiz do <code>data-application-gov-hub</code>, rode <code>docker compose up -d postgres</code>. Tenha à mão a senha <code>POSTGRES_PASSWORD</code> do seu <code>.env</code>.</div>

## Como usar este notebook

Execute as células **em ordem**, de cima para baixo: clique na célula e aperte <kbd>Shift</kbd> + <kbd>Enter</kbd>. Cada passo tem uma explicação curta, o código e um quadro com o **resultado esperado** para você comparar.

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">💡 Quadros roxos</b><br>explicam um conceito.</div>
<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">⚠️ Quadros rosa</b><br>avisam de algo que costuma dar errado.</div>
<div style="border-left:5px solid #0A005A;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#0A005A">📂 Quadros azul-marinho</b><br>mostram onde a mesma ideia aparece no repositório <code>data-application-gov-hub</code>.</div>

**Onde rodar.** Qualquer uma destas opções serve:

| Opção | Como |
|---|---|
| JupyterLab na sua máquina | `pip install jupyterlab` e depois `jupyter lab` na pasta do arquivo |
| VS Code | Abra o `.ipynb` com a extensão **Jupyter** e escolha um Python 3.10 ou mais novo |
| Jupyter da aplicação | Com o `data-application-gov-hub` no ar, coloque o arquivo na pasta `notebooks/` e abra `http://localhost:8888` |

##  Preparação

Instale os pacotes e carregue as funções de apoio.

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">⚠️ Versão do SQLAlchemy</b><br>O pandas 3 só reconhece a conexão do SQLAlchemy a partir da versão 2.0.36. Com uma versão anterior, o <code>to_sql</code> falha com <code>'Engine' object has no attribute 'cursor'</code>.</div>

In [ ]:
%pip install -q httpx psycopg2-binary pandas "sqlalchemy>=2.0.36"

In [ ]:
from IPython.display import HTML, display


def mostrar(tipo, titulo, detalhe=""):
    """Mostra um quadro colorido com o resultado de uma conferência."""
    cor, simbolo = {"ok": ("#613EFF", "✅"), "erro": ("#F9006F", "❌"), "info": ("#0A005A", "ℹ️")}[tipo]
    display(HTML(
        f'<div style="border-left:5px solid {cor};background:#F2F1F6;padding:10px 14px;'
        f'border-radius:8px;margin:6px 0;color:#202020"><b style="color:{cor}">{simbolo} {titulo}</b>'
        + (f"<br>{detalhe}" if detalhe else "") + "</div>"))


def conferir(condicao, certo, errado):
    """Mostra um quadro de acerto ou de erro, conforme a condição."""
    mostrar("ok" if condicao else "erro", certo if condicao else errado)


mostrar("ok", "Ambiente pronto", "Pode seguir para o passo 1.")

### Carregar a coleta

O notebook anterior deixou `pncp_dispensas_2026_06.json` nesta pasta. Se o arquivo não estiver aqui, a célula coleta de novo (leva uns 15 segundos).

In [ ]:
import json
import time
from pathlib import Path

import httpx

ARQUIVO = Path("pncp_dispensas_2026_06.json")


def coletar_de_novo():
    url = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
    filtro = {"dataInicial": "20260601", "dataFinal": "20260630",
              "codigoModalidadeContratacao": 8, "cnpj": "00394494000136"}
    registros, pagina, total = [], 1, 1
    with httpx.Client(timeout=60) as cliente:
        while pagina <= total:
            resposta = cliente.get(url, params={**filtro, "pagina": pagina})
            if resposta.status_code == 429:
                time.sleep(10)
                continue
            resposta.raise_for_status()
            corpo = resposta.json()
            registros.extend(corpo["data"])
            total = corpo["totalPaginas"]
            pagina += 1
            time.sleep(1)
    ARQUIVO.write_text(json.dumps(registros, ensure_ascii=False), encoding="utf-8")
    return registros


registros = json.loads(ARQUIVO.read_text(encoding="utf-8")) if ARQUIVO.exists() else coletar_de_novo()
conferir(len(registros) == 61, f"{len(registros)} registros carregados",
         f"{len(registros)} registros: o órgão pode ter publicado algo novo; siga mesmo assim")

##  Passo 1 de 6 · Conectar ao PostgreSQL

O endereço do banco depende de onde este notebook roda:

| Onde o notebook roda | Endereço do banco |
|---|---|
| Na sua máquina (JupyterLab ou VS Code) | `localhost`, porta `5432`, publicada pelo `docker-compose.yml` |
| No Jupyter da aplicação (`localhost:8888`) | `postgres`, o nome do serviço dentro da rede do Docker |

A célula descobre sozinha. A senha é pedida com `getpass`, que não a mostra nem a salva no notebook.

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">⚠️ Nunca escreva uma senha no código</b><br>Se a variável de ambiente <code>POSTGRES_PASSWORD</code> existir, ela é usada; senão, o notebook pergunta.</div>

In [ ]:
import getpass
import os
import socket

import psycopg2


def endereco_do_banco():
    try:
        socket.gethostbyname("postgres")
        return "postgres"
    except OSError:
        return "localhost"


HOST = os.environ.get("PGHOST") or endereco_do_banco()
SENHA = os.environ.get("POSTGRES_PASSWORD") or getpass.getpass("POSTGRES_PASSWORD do .env: ")
conexao = psycopg2.connect(host=HOST, port=5432, dbname="postgres", user="postgres", password=SENHA)
mostrar("ok", f"Conectado ao PostgreSQL {conexao.server_version // 10000} em {HOST}:5432")

<div style="border-left:5px solid #BE006E;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#BE006E">🧭 Deu erro de conexão?</b><br><code>Connection refused</code>: o banco não está no ar (<code>docker compose up -d postgres</code>). <code>password authentication failed</code>: confira a senha no <code>.env</code>.</div>

##  Passo 2 de 6 · Parte A: psycopg2, com upsert pela chave

Você escolhe as colunas, cria a tabela e escreve o `insert`. Seguindo a regra da camada raw, os valores ficam como **texto**.

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">💡 Upsert</b><br>A <code>primary key</code> define a identidade da linha. O <code>on conflict ... do update</code> troca a linha existente em vez de criar outra. É a idempotência da aula de cargas, agora no seu código.</div>

In [ ]:
from psycopg2.extras import execute_values

linhas = [
    (
        r["numeroControlePNCP"],
        r["orgaoEntidade"]["razaoSocial"],
        r["unidadeOrgao"]["ufSigla"],
        str(r["valorTotalEstimado"]),
        r["dataPublicacaoPncp"],
    )
    for r in registros
]

with conexao, conexao.cursor() as cursor:
    cursor.execute("create schema if not exists formacao")
    cursor.execute("""
        create table if not exists formacao.pncp_psycopg (
            numero_controle_pncp text primary key,
            orgao text,
            uf text,
            valor_estimado text,
            data_publicacao text
        )
    """)
    execute_values(cursor, """
        insert into formacao.pncp_psycopg values %s
        on conflict (numero_controle_pncp) do update set
            orgao = excluded.orgao,
            uf = excluded.uf,
            valor_estimado = excluded.valor_estimado,
            data_publicacao = excluded.data_publicacao
    """, linhas)
    cursor.execute("select count(*) from formacao.pncp_psycopg")
    total_psycopg = cursor.fetchone()[0]

conferir(total_psycopg == len(registros), f"formacao.pncp_psycopg tem {total_psycopg} linhas",
         f"formacao.pncp_psycopg tem {total_psycopg} linhas, esperado {len(registros)}")

**Rode a célula de novo.** O total continua o mesmo: o upsert trocou as linhas em vez de duplicá-las.

`with conexao` confirma a transação no fim do bloco, ou desfaz tudo se der erro no meio. `execute_values` manda as linhas em um comando só, em vez de uma ida ao banco por linha.

##  Passo 3 de 6 · Parte B: pandas, rápido e com armadilhas

O pandas transforma a lista de dicionários em tabela com uma linha.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

motor = create_engine(f"postgresql+psycopg2://postgres:{SENHA}@{HOST}:5432/postgres")
tabela = pd.json_normalize(registros, sep="__")
print(tabela.shape)
print(tabela["valorTotalHomologado"].isna().sum(), "registros sem valor homologado")
tabela[["numeroControlePNCP", "orgaoEntidade__razaoSocial", "unidadeOrgao__ufSigla", "valorTotalEstimado"]].head()

<details>
<summary><b>✔️ Resultado esperado</b> (clique para abrir)</summary>

```text
(61, 55)
8 registros sem valor homologado
```

</details>

`json_normalize` abriu os objetos internos em colunas (`orgaoEntidade__razaoSocial`): 55 colunas. Agora grave algumas delas, **duas vezes**:

In [ ]:
colunas = ["numeroControlePNCP", "orgaoEntidade__razaoSocial", "unidadeOrgao__ufSigla",
           "valorTotalEstimado", "valorTotalHomologado", "dataPublicacaoPncp"]

tabela[colunas].to_sql("pncp_pandas", motor, schema="formacao", if_exists="replace", index=False)
tabela[colunas].to_sql("pncp_pandas", motor, schema="formacao", if_exists="append", index=False)

total_pandas = pd.read_sql("select count(*) from formacao.pncp_pandas", motor).iloc[0, 0]
conferir(total_pandas == len(registros), f"{total_pandas} linhas",
         f"{total_pandas} linhas: o to_sql não faz upsert, cada gravação acrescenta tudo de novo")

<div style="border-left:5px solid #F9006F;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#F9006F">⚠️ Sem upsert</b><br><code>if_exists="append"</code> acrescenta sempre. <code>if_exists="replace"</code> apaga a tabela e cria outra, perdendo chaves, índices e permissões. E os nomes ficam com aspas (<code>"valorTotalEstimado"</code>), que você terá de usar nas consultas.</div>

Agora tente gravar **todas** as colunas:

In [ ]:
try:
    tabela.to_sql("pncp_pandas_tudo", motor, schema="formacao", if_exists="replace", index=False)
    mostrar("ok", "Gravou tudo")
except Exception as erro:
    causa = erro.__cause__ or erro
    mostrar("erro", "Não gravou", f"<code>{type(causa).__name__}: {str(causa).splitlines()[0]}</code>")

<details>
<summary><b>✔️ Resultado esperado</b> (clique para abrir)</summary>

```text
❌ Não gravou
ProgrammingError: (psycopg2.ProgrammingError) can't adapt type 'dict'
```

</details>

A coluna `fontesOrcamentarias` é uma **lista de objetos**, e o PostgreSQL não sabe guardar um dicionário Python numa coluna comum. Toda API real tem algum campo assim.

##  Passo 4 de 6 · Parte C: o cliente do repositório

O `ClientPostgresDB`, em [`plugins/cliente_postgres.py`](https://github.com/GovHub-br/data-application-gov-hub/blob/main/airflow_lappis/plugins/cliente_postgres.py), é o que as DAGs usam para gravar. Ele resolve as armadilhas da parte B: achata objetos e listas, cria as colunas como texto e faz upsert.

A célula abaixo procura a pasta `airflow_lappis/plugins` do repositório. Ela é encontrada se este notebook estiver dentro do `data-application-gov-hub`, por exemplo em `notebooks/`, rodando na sua máquina.

<div style="border-left:5px solid #BE006E;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#BE006E">🧭 Rodando no Jupyter da aplicação?</b><br>Ele não enxerga as pastas do repositório. Use o caminho alternativo que a célula mostra: o mesmo código, dentro do container do Airflow.</div>

In [ ]:
import sys

candidatos = [Path.cwd().parent / "airflow_lappis" / "plugins", Path.cwd() / "airflow_lappis" / "plugins"]
PLUGINS = next((p for p in candidatos if (p / "cliente_postgres.py").exists()), None)

if PLUGINS:
    sys.path.insert(0, str(PLUGINS))
    from cliente_postgres import ClientPostgresDB

    db = ClientPostgresDB(f"dbname=postgres user=postgres password={SENHA} host={HOST} port=5432")
    for _ in range(2):   # duas vezes, de propósito
        db.insert_data(
            registros,
            table_name="pncp_cliente",
            conflict_fields=["numeroControlePNCP"],
            primary_key=["numeroControlePNCP"],
            schema="formacao",
        )
    [(total_cliente, com_nan)] = db.execute_query(
        "select count(*), count(*) filter (where valortotalhomologado = 'NaN') from formacao.pncp_cliente")
    conferir(total_cliente == len(registros), f"formacao.pncp_cliente tem {total_cliente} linhas, mesmo gravando duas vezes",
             f"formacao.pncp_cliente tem {total_cliente} linhas")
    mostrar("info", f"{com_nan} valores homologados gravados como o texto 'NaN'",
            "O json_normalize transforma None em NaN, e a coluna TEXT guarda o texto. A bronze do dbt trata isso.")
else:
    total_cliente = None
    mostrar("info", "Pasta airflow_lappis/plugins não encontrada",
            "Na raiz do <code>data-application-gov-hub</code>, rode:<br>"
            "<code>docker compose cp pncp_dispensas_2026_06.json airflow:/tmp/</code><br>"
            "<code>docker compose exec airflow python</code><br>"
            "e cole o código da parte C da aula, que usa <code>get_postgres_conn()</code> em vez da senha.")

<details>
<summary><b>✔️ Resultado esperado</b> (clique para abrir)</summary>

```text
✅ formacao.pncp_cliente tem 61 linhas, mesmo gravando duas vezes
ℹ️ 8 valores homologados gravados como o texto 'NaN'
```

</details>

Sem você escrever SQL, o `insert_data`:

| Fez | Como |
|---|---|
| Criou schema e tabela | 55 colunas, todas `TEXT`, com chave primária em `numerocontrolepncp` |
| Achatou objetos e listas | `orgaoEntidade.cnpj` virou `orgaoentidade__cnpj`; listas viraram texto |
| Fez upsert | `on conflict (numeroControlePNCP) do update` |

Nas DAGs, a conexão vem de `get_postgres_conn()`, que lê a Connection `postgres_default` do Airflow. Aqui você passou a conexão à mão porque o notebook roda fora do Airflow.

##  Passo 5 de 6 · Parte D: outro banco, o mesmo padrão

Bancos relacionais falam a mesma língua em Python (a especificação DB-API): conectar, abrir um cursor, executar com parâmetros, confirmar. Veja no **SQLite**, que grava num arquivo e já vem com o Python.

In [ ]:
import sqlite3

with sqlite3.connect("estudo.db") as banco:
    banco.execute("""
        create table if not exists pncp (
            numero_controle_pncp text primary key,
            orgao text,
            valor_estimado real
        )
    """)
    banco.executemany("""
        insert into pncp values (?, ?, ?)
        on conflict (numero_controle_pncp) do update set valor_estimado = excluded.valor_estimado
    """, [(r["numeroControlePNCP"], r["orgaoEntidade"]["razaoSocial"], r["valorTotalEstimado"]) for r in registros])
    total_sqlite, soma = banco.execute("select count(*), round(sum(valor_estimado), 2) from pncp").fetchone()

mostrar("ok", f"estudo.db: {total_sqlite} linhas, R$ {soma:,.2f} estimados")

<details>
<summary><b>✔️ Resultado esperado</b> (clique para abrir)</summary>

```text
✅ estudo.db: 61 linhas, R$ 4,599,247.08 estimados
```

</details>

Mudam o `connect`, o marcador de parâmetro (`?` em vez de `%s`) e alguns tipos. A chave, o upsert e a conferência são os mesmos.

<div style="border-left:5px solid #0A005A;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#0A005A">📂 Outro banco como fonte</b><br>No Gov Hub também existe o caminho inverso: a DAG <code>dags/data_ingest/sisbolsas/sisbolsas_to_postgres_ingest_dag.py</code> lê tabelas de um <b>SQL Server</b> com <code>ClientSQLServerDB.fetch_table_all</code> e grava com o mesmo <code>insert_data</code>. Para a ingestão, outro banco é só mais uma fonte.</div>

##  Passo 6 de 6 · Conferir tudo

As quatro formas, lado a lado:

In [ ]:
resumo = pd.DataFrame([
    ("A · psycopg2", "formacao.pncp_psycopg", total_psycopg, "upsert pela chave"),
    ("B · pandas", "formacao.pncp_pandas", total_pandas, "sem upsert: duplicou"),
    ("C · insert_data", "formacao.pncp_cliente", total_cliente, "upsert e listas tratadas"),
    ("D · SQLite", "estudo.db", total_sqlite, "mesmo padrão, outro banco"),
], columns=["forma", "tabela", "linhas", "observação"])
conferir(total_pandas != len(registros), "A tabela mostra a diferença: só o pandas duplicou",
         "Todas as formas ficaram com o mesmo total")
resumo

<details>
<summary><b>✔️ Resultado esperado</b> (clique para abrir)</summary>

```text
forma             tabela                  linhas  observação
A · psycopg2      formacao.pncp_psycopg       61  upsert pela chave
B · pandas        formacao.pncp_pandas       122  sem upsert: duplicou
C · insert_data   formacao.pncp_cliente       61  upsert e listas tratadas
D · SQLite        estudo.db                   61  mesmo padrão, outro banco
```

</details>

##  Desafios

**1.** Corrija a parte B para não duplicar: grave com `if_exists="replace"` em uma tabela temporária e faça o upsert em SQL a partir dela.

**2.** Na tabela `formacao.pncp_cliente`, quais colunas vieram de objetos internos? (Dica: procure `__` em `information_schema.columns`.)

<details>
<summary><b>✔️ Ver uma resposta para o desafio 2</b> (clique para abrir)</summary>

```text
pd.read_sql("""
    select column_name from information_schema.columns
    where table_schema = 'formacao' and table_name = 'pncp_cliente'
      and position('__' in column_name) > 0
""", motor)
```

</details>

In [ ]:
# Escreva aqui as suas respostas

### Limpar o estudo

As tabelas deste notebook servem só para comparar. Quando terminar, apague-as:

In [ ]:
with conexao, conexao.cursor() as cursor:
    cursor.execute("drop table if exists formacao.pncp_psycopg, formacao.pncp_pandas, "
                   "formacao.pncp_pandas_tudo, formacao.pncp_cliente")
conexao.close()
mostrar("ok", "Tabelas de estudo apagadas", "O arquivo estudo.db pode ser apagado da pasta.")

## O que você aprendeu

- Gravar sem duplicar exige uma chave e um upsert; o `to_sql` do pandas não faz isso.
- APIs reais trazem objetos e listas; a raw precisa de uma regra para guardá-los.
- O `insert_data` do repositório encapsula essas regras, e por isso as DAGs o usam.
- Trocar de banco muda a conexão, não o raciocínio.

<div style="background:#F2F1F6;border-radius:16px;padding:22px 26px;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif;border:2px solid #613EFF">
<div style="color:#BE006E;font-weight:700;font-size:.8em;letter-spacing:.06em">PRÓXIMO PASSO</div>
<div style="color:#0A005A;font-size:1.3em;font-weight:800;margin:6px 0">Prática: orquestrar a coleta com o Airflow</div>
<div style="color:#333">Baixe o notebook <code>a aula da trilha</code> na mesma aula da trilha, ou leia a versão em texto em
<a href="https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-ingestao-bancos.md">gov-hub.io/capacitacao</a>.</div>
</div>

<p style="text-align:center;color:#666;font-size:.85em;margin-top:18px;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">Capacitação Gov Hub · Trilha de Engenharia de Dados · material aberto, feito para ser executado na sua máquina.</p>